# Phase 3 — Your First Vector Database with Chroma

**Goal:** learn how to create a collection, store text chunks and embeddings, attach metadata, embed a query, run top-K similarity search, and inspect the results.

This notebook uses:
- **Chroma** — a document-oriented vector database with metadata and local persistence.
- **Sentence Transformers** — an open-source embedding model that runs locally.
- **A small sample knowledge base** — no API key or paid model required.

> **Important:** Chroma's `distance` is a distance value, not a similarity percentage. With cosine distance, **smaller is better**.

## 0. Install the packages with `uv`

Run these commands in your terminal from the folder where you want to keep this notebook. If you already have a `uv` project, you can skip `uv init`.

```bash
uv init vector-db-learning
cd vector-db-learning
uv add chromadb sentence-transformers jupyter
uv run jupyter lab
```

Open this notebook in Jupyter. The first run may download the embedding model (roughly a few dozen MB) and can take a little time. The model then runs locally.

If you prefer VS Code, install its **Jupyter** extension and select the Python environment created by `uv`.

## 1. Import libraries and load a local embedding model

An **embedding** is a list of numbers representing the meaning of text. Texts with similar meanings tend to have vectors that are close together.

We use `all-MiniLM-L6-v2`, a compact Sentence Transformers model. It is free to use locally; the first download requires internet access.

In [3]:
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


## 2. Prepare example document chunks

In a real RAG pipeline, you would extract text from PDFs, Word files, or web pages and split it into chunks. Here, each string is already a chunk.

Each chunk has:
- a unique **ID**
- the chunk **text**
- **metadata** such as filename, page, and section

Keep chunks small enough to retrieve a focused passage, but large enough to preserve context.

In [4]:
chunks = [
    {
        "id": "hr-001",
        "text": "Employees can request annual leave through the HR portal. Their manager must approve the request before the leave is confirmed.",
        "metadata": {"filename": "employee_handbook.pdf", "page": 4, "section": "Annual Leave"},
    },
    {
        "id": "hr-002",
        "text": "Unused annual leave is handled according to the company's leave policy and applicable local employment rules.",
        "metadata": {"filename": "employee_handbook.pdf", "page": 5, "section": "Leave Balance"},
    },
    {
        "id": "it-001",
        "text": "To reset a forgotten password, open the company identity portal and choose Forgot password. Complete the verification steps.",
        "metadata": {"filename": "it_support_guide.pdf", "page": 2, "section": "Password Reset"},
    },
    {
        "id": "it-002",
        "text": "Multi-factor authentication adds a second verification step, such as an authenticator app prompt, when signing in.",
        "metadata": {"filename": "it_security_guide.pdf", "page": 3, "section": "Multi-Factor Authentication"},
    },
    {
        "id": "dev-001",
        "text": "A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector.",
        "metadata": {"filename": "rag_basics.pdf", "page": 1, "section": "Vector Databases"},
    },
    {
        "id": "dev-002",
        "text": "In retrieval-augmented generation, the application retrieves relevant document chunks and provides them as context to a language model.",
        "metadata": {"filename": "rag_basics.pdf", "page": 2, "section": "RAG Workflow"},
    },
    {
        "id": "dev-003",
        "text": "Chunk metadata can identify the source filename, page number, section title, and other filtering fields.",
        "metadata": {"filename": "rag_basics.pdf", "page": 3, "section": "Metadata"},
    },
]

texts = [item["text"] for item in chunks]
ids = [item["id"] for item in chunks]
metadatas = [item["metadata"] for item in chunks]

print(f"Chunks: {len(chunks)}")
print("Example metadata:", metadatas[0])

Chunks: 7
Example metadata: {'filename': 'employee_handbook.pdf', 'page': 4, 'section': 'Annual Leave'}


## 3. Create a Chroma collection (the vector index)

A **collection** is similar to a table or index that holds records, metadata, and embeddings.

This notebook uses a persistent local folder named `chroma_db`. Your data can remain there between notebook sessions. We delete the tutorial collection first so you can rerun the notebook without duplicate/stale sample records.

In [13]:
DB_PATH = str(Path("chroma_db").resolve())
COLLECTION_NAME = "phase3_learning"

client = chromadb.PersistentClient(path=DB_PATH)

# Reset only this tutorial collection so rerunning the notebook is predictable.
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print("Database folder:", DB_PATH)
print("Collection:", collection.name)

Database folder: F:\MANIKANDAN-Office\RAG\notebooks\chroma_db
Collection: phase3_learning


## 4. Create embeddings and add chunks, IDs, and metadata

We create one embedding per chunk. `model.encode()` returns numeric vectors. Then we store those vectors in Chroma along with the original text and metadata.

In this example, we calculate embeddings ourselves so the vector creation step is visible. Chroma can also use an embedding function for you.

In [14]:
embeddings = model.encode(texts, normalize_embeddings=True).tolist()

print("Number of embeddings:", len(embeddings))
print("Dimensions of one embedding:", len(embeddings[0]))
print("First 8 values:", embeddings[0][:8])

collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings,
)

print("Records stored:", collection.count())

Number of embeddings: 7
Dimensions of one embedding: 384
First 8 values: [0.012330854311585426, 0.05176151543855667, 0.005960110109299421, 0.038374319672584534, 0.08637136220932007, 0.036474116146564484, -0.055599644780159, -0.10203830897808075]
Records stored: 7


## 5. Embed a question and run top-K similarity search

A query must be embedded using the **same embedding model** as the stored chunks. We pass its vector to Chroma and ask for the `K` nearest records.

Try changing the question to:
- `How can I change my password?`
- `What does a vector database do?`
- `How does RAG use retrieved documents?`

In [15]:
query_text = "what is vector database?"

query_embedding = model.encode(
    [query_text],
    normalize_embeddings=True,
).tolist()[0]

print("Question:", query_text)
print("Query vector dimensions:", len(query_embedding))

Question: what is vector database?
Query vector dimensions: 384


## 6. Inspect returned text, scores, and metadata

Chroma returns nested lists because a query can contain multiple query vectors. We submitted one query, so inspect result group `[0]`.

The `distances` value is **cosine distance** in this collection:
- closer to `0` generally means more similar
- larger distance means less similar
- it is **not** a percentage or a calibrated confidence score

In [16]:
def search_and_print(question: str, k: int = 3):
    q_embedding = model.encode(
        [question],
        normalize_embeddings=True,
    ).tolist()[0]

    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=min(k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    print(f"\nQuestion: {question!r} | K={k}")
    print("=" * 90)

    for rank, (result_id, document, metadata, distance) in enumerate(
        zip(
            results["ids"][0],
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ),
        start=1,
    ):
        print(f"Rank: {rank} | ID: {result_id} | cosine distance: {distance:.4f}")
        print(f"Source: {metadata.get('filename')} | Page: {metadata.get('page')} | Section: {metadata.get('section')}")
        print(f"Text: {document}")
        print("-" * 90)

    return results

results = search_and_print(query_text, k=3)


Question: 'what is vector database?' | K=3
Rank: 1 | ID: dev-001 | cosine distance: 0.2293
Source: rag_basics.pdf | Page: 1 | Section: Vector Databases
Text: A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector.
------------------------------------------------------------------------------------------
Rank: 2 | ID: dev-002 | cosine distance: 0.7611
Source: rag_basics.pdf | Page: 2 | Section: RAG Workflow
Text: In retrieval-augmented generation, the application retrieves relevant document chunks and provides them as context to a language model.
------------------------------------------------------------------------------------------
Rank: 3 | ID: dev-003 | cosine distance: 0.8146
Source: rag_basics.pdf | Page: 3 | Section: Metadata
Text: Chunk metadata can identify the source filename, page number, section title, and other filtering fields.
-----------------------------------------------------------------------------------------

## 7. Experiment with K = 1, 3, and 5

`K` controls how many candidate chunks are returned. It does **not** change the embeddings or the stored index.

In [9]:
for k in [1, 3, 5]:
    search_and_print(query_text, k=k)


Question: 'what is vector database?' | K=1
Rank: 1 | ID: dev-001 | cosine distance: 0.2293
Source: rag_basics.pdf | Page: 1 | Section: Vector Databases
Text: A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector.
------------------------------------------------------------------------------------------

Question: 'what is vector database?' | K=3
Rank: 1 | ID: dev-001 | cosine distance: 0.2293
Source: rag_basics.pdf | Page: 1 | Section: Vector Databases
Text: A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector.
------------------------------------------------------------------------------------------
Rank: 2 | ID: dev-002 | cosine distance: 0.7611
Source: rag_basics.pdf | Page: 2 | Section: RAG Workflow
Text: In retrieval-augmented generation, the application retrieves relevant document chunks and provides them as context to a language model.
-------------------------------

### What should you observe?

- **K = 1:** only the closest chunk is returned. This can be precise, but may omit useful context.
- **K = 3:** gives a few candidates and is a useful starting point for many small RAG experiments.
- **K = 5:** returns more context, but some results may be less relevant.
- Inspect the source metadata to see which file, page, and section supplied each result.
- Do not assume every returned chunk is relevant just because it appears in the top K.

## 8. Filter by metadata

Metadata lets you restrict retrieval to a source or section. For example, this query only searches chunks from `rag_basics.pdf`.

The `where` filter operates on metadata fields, not on semantic meaning.

In [17]:
question = "What is a vector database?"
q_embedding = model.encode([question], normalize_embeddings=True).tolist()[0]

filtered_results = collection.query(
    query_embeddings=[q_embedding],
    n_results=3,
    where={"filename": "rag_basics.pdf"},
    include=["documents", "metadatas", "distances"],
)

for rank, (doc_id, document, metadata, distance) in enumerate(
    zip(
        filtered_results["ids"][0],
        filtered_results["documents"][0],
        filtered_results["metadatas"][0],
        filtered_results["distances"][0],
    ),
    start=1,
):
    print(f"{rank}. {doc_id} | distance={distance:.4f} | {metadata}")
    print(document, "\n")

1. dev-001 | distance=0.2152 | {'filename': 'rag_basics.pdf', 'section': 'Vector Databases', 'page': 1}
A vector database stores embedding vectors and can retrieve records whose vectors are close to a query vector. 

2. dev-002 | distance=0.7673 | {'page': 2, 'section': 'RAG Workflow', 'filename': 'rag_basics.pdf'}
In retrieval-augmented generation, the application retrieves relevant document chunks and provides them as context to a language model. 

3. dev-003 | distance=0.8115 | {'filename': 'rag_basics.pdf', 'page': 3, 'section': 'Metadata'}
Chunk metadata can identify the source filename, page number, section title, and other filtering fields. 



## 9. Persistence: reopen the collection

Because we used `PersistentClient`, Chroma writes data to the `chroma_db` folder. The next cell creates a new client and opens the same collection.

Run it after the insert step. If you restart the kernel, run the imports/model setup first, then this cell to verify persistence without recreating the collection.

In [18]:
client_again = chromadb.PersistentClient(path=DB_PATH)
collection_again = client_again.get_collection(COLLECTION_NAME)

print("Records after reopening:", collection_again.count())
print("Collection name:", collection_again.name)

Records after reopening: 7
Collection name: phase3_learning


## 10. Mini exercises

Try these yourself before looking at the suggested answers.

1. Change the query to `Who approves annual leave?` and compare K=1, 3, and 5.
2. Add one new chunk with a unique ID, text, and metadata; embed it and store it with `collection.add(...)`.
3. Search only within `it_support_guide.pdf` using a `where` filter.
4. Print just the IDs and distances in a compact list.
5. Explain in your own words the difference between **embedding**, **vector database**, **metadata**, **top-K**, and **distance**.

### Example: add one more chunk

Uncomment and run this cell after adapting the content. IDs must be unique within the collection.

In [12]:
# extra_text = "Employees should contact the IT help desk if account verification fails."
# extra_embedding = model.encode([extra_text], normalize_embeddings=True).tolist()
#
# collection.add(
#     ids=["it-003"],
#     documents=[extra_text],
#     metadatas=[{
#         "filename": "it_support_guide.pdf",
#         "page": 4,
#         "section": "Account Help",
#     }],
#     embeddings=extra_embedding,
# )
#
# print("Updated record count:", collection.count())

## Key takeaways

- **Embedding model:** converts text into vectors.
- **Vector database:** stores vectors and supports nearest-neighbor retrieval.
- **ID:** stable identifier for a stored chunk.
- **Metadata:** structured source details useful for filtering and citations.
- **Top-K:** number of nearest results to return.
- **Cosine distance:** a ranking distance here; lower is closer.
- **RAG connection:** retrieve useful chunks first, then pass their text to a language model as context.

**Next phase:** load a real PDF, split it into chunks, store filename/page/section metadata, and use the retrieved passages as context in a simple RAG question-answering pipeline.